# Elicitation; malicious links and files  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 1.1.b: Malicious link and file indicator detector

**Skill:** Analyze Risk · **Maps to:** 1.1.A, 1.1.C · **Language:** Python 3 (pytest)

## Problem
Build the analyzer a phishing-awareness tool would use to flag suspicious URLs and filenames.
Given a single string that is either a URL or a filename, return which malicious indicators
it exhibits.

Four indicator categories are defined:

- `"lookalike-domain"`, the URL hostname resembles a known brand but is not the real domain
  (typosquatting rules defined in the Reference section below)
- `"ip-url"`, the URL uses a raw IPv4 address as its host instead of a domain name
- `"dangerous-extension"`, the filename (or URL path) ends in an executable or script extension
- `"mismatched-link"`, the URL contains a brand keyword in a path or subdomain while the
  actual registrable domain belongs to a different host (defined precisely in the Reference section below)

## Function contract
Implement `solve(url_or_filename: str) -> list[str]` in the code editor
(see the Reference section below).

Return the detected indicator labels as a **sorted, de-duplicated** list.

## Input
A single string, either a URL (starting with `http://` or `https://`) or a bare filename.

## Output
A sorted list of zero or more indicator labels drawn from
`{"dangerous-extension", "ip-url", "lookalike-domain", "mismatched-link"}`.

## Examples
```
"http://paypa1.com/verify" -> ["lookalike-domain"]
"http://192.168.4.99/reset" -> ["ip-url"]
"invoice.pdf.exe" -> ["dangerous-extension"]
"http://bankofamerica.com.secure-login.xyz" -> ["lookalike-domain", "mismatched-link"]
"http://shop.example/track/44821" -> []
"http://dropbx.com/s/Q3.scr" -> ["dangerous-extension", "lookalike-domain"]
```

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(url_or_filename: str) -> list[str]
```

### Input (`url_or_filename`)
A single string. If it starts with `http://` or `https://` it is a URL; otherwise it is treated
as a bare filename.

### Output (`result`)
A **sorted, de-duplicated** `list[str]` drawn from
`{"dangerous-extension", "ip-url", "lookalike-domain", "mismatched-link"}`.
Empty list if no indicator is present.

### Detection rules (all case-insensitive)

#### `ip-url`
The hostname portion of the URL matches an IPv4 address: four decimal octets separated by dots,
optionally followed by a port (e.g. `192.168.4.99`, `10.0.0.1:8080`).
Detection regex (applied to hostname only): `^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}(:\d+)?$`

#### `lookalike-domain`
The registrable domain (last two labels of the hostname, e.g. `paypa1.com`) matches any entry in
the LOOKALIKE_DOMAINS set below.

LOOKALIKE_DOMAINS (exact match after lowercasing the registrable domain):
```
paypa1.com, arnazon.com, dropbx.com, g00gle.com, faceb00k.com,
rn1crosoft.com, rnicrosott.com, rnicrosoft.com, linkedln.com,
app1e.com, twiitter.com, lnstagram.com
```

#### `dangerous-extension`
The final extension of the string (extracted from the last path segment, or from the bare
filename if no URL) is in the DANGEROUS_EXTENSIONS set.

DANGEROUS_EXTENSIONS:
```
.exe, .scr, .bat, .cmd, .ps1, .vbs, .vbe, .js, .jse, .wsf, .wsh,
.jar, .pif, .com, .hta, .msi, .dll, .cpl, .reg
```

For double-extension filenames like `report.pdf.exe`, the final extension is `.exe`, that
extension alone triggers the indicator.

#### `mismatched-link`
Applies only to URLs (not bare filenames).
A URL is a mismatched-link if:
1. The registrable domain is NOT in the TRUSTED_DOMAINS set below, AND
2. The URL contains at least one TRUSTED_BRAND keyword anywhere in the full URL string
   (scheme + host + path) EXCLUDING the registrable domain itself.

This models the pattern `http://bankofamerica.com.evil.net/...` where `bankofamerica.com`
appears as a subdomain of `evil.net`.

TRUSTED_DOMAINS (the real registrable domains):
```
paypal.com, amazon.com, dropbox.com, google.com, facebook.com,
microsoft.com, linkedin.com, apple.com, twitter.com, instagram.com,
bankofamerica.com, chase.com, wellsfargo.com, company.example,
shop.example, meridian-corp.example
```

TRUSTED_BRAND keywords (substring match in the full URL, case-insensitive):
```
paypal, amazon, dropbox, google, facebook, microsoft, linkedin,
apple, twitter, instagram, bankofamerica, chase, wellsfargo
```

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Matching is case-insensitive throughout.
- Output is always sorted ascending and de-duplicated.
- A single input may trigger multiple indicators simultaneously.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 1.1.b. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
import re
from typing import List
from urllib.parse import urlparse

# --- constants from the Reference section below ---

LOOKALIKE_DOMAINS = {
    "paypa1.com", "arnazon.com", "dropbx.com", "g00gle.com", "faceb00k.com",
    "rn1crosoft.com", "rnicrosott.com", "rnicrosoft.com", "linkedln.com",
    "app1e.com", "twiitter.com", "lnstagram.com",
}

DANGEROUS_EXTENSIONS = {
    ".exe", ".scr", ".bat", ".cmd", ".ps1", ".vbs", ".vbe", ".js", ".jse",
    ".wsf", ".wsh", ".jar", ".pif", ".com", ".hta", ".msi", ".dll", ".cpl", ".reg",
}

TRUSTED_DOMAINS = {
    "paypal.com", "amazon.com", "dropbox.com", "google.com", "facebook.com",
    "microsoft.com", "linkedin.com", "apple.com", "twitter.com", "instagram.com",
    "bankofamerica.com", "chase.com", "wellsfargo.com", "company.example",
    "shop.example", "meridian-corp.example",
}

TRUSTED_BRANDS = [
    "paypal", "amazon", "dropbox", "google", "facebook", "microsoft", "linkedin",
    "apple", "twitter", "instagram", "bankofamerica", "chase", "wellsfargo",
]

_IP_RE = re.compile(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}(:\d+)?$')


def _registrable_domain(hostname: str) -> str:
    """Return last two labels of a hostname, lowercased (e.g. 'evil.net' from 'foo.evil.net')."""
    parts = hostname.lower().split(".")
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return hostname.lower()


def _final_extension(s: str) -> str:
    """Return the final dot-extension of a string (path segment or filename), lowercased."""
    # take the last path segment
    segment = s.rstrip("/").split("/")[-1].split("?")[0].split("#")[0]
    dot_idx = segment.rfind(".")
    if dot_idx == -1:
        return ""
    return segment[dot_idx:].lower()


def solve(url_or_filename: str) -> List[str]:
    """Return sorted, de-duplicated malicious indicators present in url_or_filename."""
    s = url_or_filename or ""
    found = set()
    is_url = s.lower().startswith("http://") or s.lower().startswith("https://")

    if is_url:
        parsed = urlparse(s)
        hostname = parsed.hostname or ""
        reg_domain = _registrable_domain(hostname)

        # ip-url
        if _IP_RE.match(parsed.netloc.lower()):
            found.add("ip-url")

        # lookalike-domain
        if reg_domain in LOOKALIKE_DOMAINS:
            found.add("lookalike-domain")

        # mismatched-link: not a trusted domain, but URL contains a trusted brand keyword
        if reg_domain not in TRUSTED_DOMAINS:
            url_lower = s.lower()
            # check keyword presence outside the registrable domain portion
            # strip reg_domain from the string and check remaining
            url_without_reg = url_lower.replace(reg_domain, "", 1)
            for brand in TRUSTED_BRANDS:
                if brand in url_without_reg:
                    found.add("mismatched-link")
                    break

        # dangerous-extension (from URL path)
        ext = _final_extension(parsed.path)
        if ext in DANGEROUS_EXTENSIONS:
            found.add("dangerous-extension")
    else:
        # bare filename — only dangerous-extension applies
        ext = _final_extension(s)
        if ext in DANGEROUS_EXTENSIONS:
            found.add("dangerous-extension")

    return sorted(found)


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------

def test_lookalike_domain():
    assert impl.solve("http://paypa1.com/verify?id=38291") == ["lookalike-domain"]


def test_ip_url():
    assert impl.solve("http://192.168.4.99/reset?user=jsmith") == ["ip-url"]


def test_dangerous_extension_filename():
    assert impl.solve("new_hire_form.pdf.exe") == ["dangerous-extension"]


def test_lookalike_and_mismatched():
    # bankofamerica.com.secure-login.xyz: reg domain is secure-login.xyz (not trusted),
    # but bankofamerica appears in the URL outside the reg domain => mismatched-link.
    # secure-login.xyz is not in LOOKALIKE_DOMAINS so no lookalike-domain here.
    result = impl.solve("http://bankofamerica.com.secure-login.xyz/stmt/2026-05.pdf")
    assert "mismatched-link" in result


def test_clean_url():
    assert impl.solve("http://shop.example/track/44821") == []


def test_lookalike_and_dangerous_extension_url():
    result = impl.solve("http://dropbx.com/s/Q3.scr")
    assert result == ["dangerous-extension", "lookalike-domain"]


# ---- hidden / edge cases -------------------------------------------------

def test_empty_string():
    assert impl.solve("") == []


def test_scr_extension_filename():
    assert impl.solve("Q3_Report.scr") == ["dangerous-extension"]


def test_ps1_extension():
    assert impl.solve("http://company.example/setup.ps1") == ["dangerous-extension"]


def test_case_insensitive_extension():
    assert impl.solve("malware.EXE") == ["dangerous-extension"]


def test_ip_url_with_port():
    assert impl.solve("http://10.0.0.1:8080/admin") == ["ip-url"]


def test_clean_filename():
    assert impl.solve("report.pdf") == []


def test_sorted_output():
    result = impl.solve("http://dropbx.com/s/Q3.scr")
    assert result == sorted(result)


def test_dedup():
    # Even if multiple rules could fire the same indicator, it appears once
    result = impl.solve("http://paypa1.com/path.exe")
    assert result == sorted(set(result))


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
